In [58]:
import torch
import torch_geometric
import pyg_lib
import torch_geometric
print(torch_geometric.typing.WITH_PYG_LIB)
import pickle
import torch_geometric.transforms as T

ModuleNotFoundError: No module named 'pyg_lib'

In [29]:
PTH = r'./picklefiles'

with open(f'{PTH}//graph_trainsmall_linkpred.pkl', 'rb') as f:
    graph = pickle.load(f)

In [30]:
graph

HeteroData(
  target_entity={
    num_nodes=5498,
    node_id=[5498],
    x=[5498, 150],
    y=[5498, 384]
  },
  t_entities={
    num_nodes=7698,
    node_id=[7698],
    x=[7698, 150]
  },
  aspect={
    num_nodes=5688,
    node_id=[5688],
    x=[5686, 150]
  },
  a_entities={
    num_nodes=53858,
    node_id=[53858],
    x=[53858, 150]
  },
  (target_entity, linked_to, aspect)={ edge_index=[2, 5498] },
  (target_entity, associated_to, t_entities)={ edge_index=[2, 11543] },
  (aspect, associated_to, a_entities)={ edge_index=[2, 542835] }
)

In [45]:
graph = T.ToUndirected()(graph)

In [47]:
graph

HeteroData(
  target_entity={
    num_nodes=5498,
    node_id=[5498],
    x=[5498, 150],
    y=[5498, 384]
  },
  t_entities={
    num_nodes=7698,
    node_id=[7698],
    x=[7698, 150]
  },
  aspect={
    num_nodes=5688,
    node_id=[5688],
    x=[5686, 150]
  },
  a_entities={
    num_nodes=53858,
    node_id=[53858],
    x=[53858, 150]
  },
  (target_entity, linked_to, aspect)={ edge_index=[2, 5498] },
  (target_entity, associated_to, t_entities)={ edge_index=[2, 11543] },
  (aspect, associated_to, a_entities)={ edge_index=[2, 542835] },
  (aspect, rev_linked_to, target_entity)={ edge_index=[2, 5498] },
  (t_entities, rev_associated_to, target_entity)={ edge_index=[2, 11543] },
  (a_entities, rev_associated_to, aspect)={ edge_index=[2, 542835] }
)

In [48]:
transform = T.RandomLinkSplit(
    disjoint_train_ratio = 0.7,   
    add_negative_train_samples = True,  
    edge_types = ("target_entity", "linked_to", "aspect"),
    rev_edge_types = ("aspect", "rev_linked_to", "target_entity")
)

In [49]:
train_data, val_data, test_data = transform(graph)

In [50]:
train_data

HeteroData(
  target_entity={
    num_nodes=5498,
    node_id=[5498],
    x=[5498, 150],
    y=[5498, 384]
  },
  t_entities={
    num_nodes=7698,
    node_id=[7698],
    x=[7698, 150]
  },
  aspect={
    num_nodes=5688,
    node_id=[5688],
    x=[5686, 150]
  },
  a_entities={
    num_nodes=53858,
    node_id=[53858],
    x=[53858, 150]
  },
  (target_entity, linked_to, aspect)={
    edge_index=[2, 1155],
    edge_label=[5390],
    edge_label_index=[2, 5390]
  },
  (target_entity, associated_to, t_entities)={ edge_index=[2, 11543] },
  (aspect, associated_to, a_entities)={ edge_index=[2, 542835] },
  (aspect, rev_linked_to, target_entity)={ edge_index=[2, 1155] },
  (t_entities, rev_associated_to, target_entity)={ edge_index=[2, 11543] },
  (a_entities, rev_associated_to, aspect)={ edge_index=[2, 542835] }
)

In [51]:
from torch_geometric.loader import LinkNeighborLoader
edge_label_index = train_data["target_entity", "linked_to", "aspect"].edge_label_index
edge_label = train_data["target_entity", "linked_to", "aspect"].edge_label

train_loader = LinkNeighborLoader(
    data = train_data,  # TODO
    num_neighbors= 20 ,  # TODO
    edge_label_index=(("target_entity", "linked_to", "aspect"), edge_label_index),
    edge_label=edge_label,
    batch_size=128,
    shuffle=True,
)

In [56]:
sampled_data = next(iter(train_loader))

ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [52]:
graph.metadata()

(['target_entity', 't_entities', 'aspect', 'a_entities'],
 [('target_entity', 'linked_to', 'aspect'),
  ('target_entity', 'associated_to', 't_entities'),
  ('aspect', 'associated_to', 'a_entities'),
  ('aspect', 'rev_linked_to', 'target_entity'),
  ('t_entities', 'rev_associated_to', 'target_entity'),
  ('a_entities', 'rev_associated_to', 'aspect')])

In [53]:
from torch_geometric.nn import SAGEConv, to_hetero


class GNN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()

        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.conv2(x)

# Our final classifier applies the dot-product between source and destination
# node embeddings to derive edge-level predictions:
class Classifier(torch.nn.Module):
    def forward(self, x_target, x_aspect, edge_label_index):
        # Convert node embeddings to edge-level representations:
        edge_feat_target = x_target[edge_label_index[0]]
        edge_feat_aspect = x_aspect[edge_label_index[1]]

        # Apply dot-product to get a prediction per supervision edge:
        return (edge_feat_target * edge_feat_aspect).sum(dim=-1)


class Model(torch.nn.Module):
    def __init__(self, inp, hidden_channels):
        super().__init__()
        # Since the dataset does not come with rich features, we also learn two
        # embedding matrices for users and movies:
        self.target_ent_lin = torch.nn.Linear(inp, hidden_channels)
        # Instantiate homogeneous GNN:
        self.gnn = GNN(hidden_channels)

        # Convert GNN model into a heterogeneous variant:
        self.gnn = to_hetero(self.gnn, metadata=graph.metadata())

        self.classifier = Classifier()

    def forward(self, data):
        x_dict = {
          "entity": self.target_ent_lin(data['target_entity'].y) + data['target_entity'].x,
          "aspect": data['aspect'].x,
          "associated target ent" : data['t_entities'].x,
          "associated aspect ent" : data['a_entities'].x
        }

        # `x_dict` holds feature matrices of all node types
        # `edge_index_dict` holds all edge indices of all edge types
        x_dict = self.gnn(x_dict, data.edge_index_dict)

        pred = self.classifier(
            x_dict["entity"],
            x_dict["aspect"],
            data["target_entity", "linked_to", "aspect"].edge_label_index,
        )

        return pred


model = Model(inp = 384, hidden_channels=150)

print(model)

Model(
  (target_ent_lin): Linear(in_features=384, out_features=150, bias=True)
  (gnn): GraphModule(
    (conv1): SAGEConv(150, 150, aggr=mean)
    (conv2): SAGEConv(150, 150, aggr=mean)
  )
  (classifier): Classifier()
)


In [55]:
import tqdm
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: '{device}'")

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1, 6):
    total_loss = total_examples = 0
    for sampled_data in tqdm.tqdm(train_loader):
        optimizer.zero_grad()
        sampled_data = sampled_data.to(device)
        pred = model(graph)
        edge_label = edge_label.to(device)
        loss = F.binary_cross_entropy_with_logits(pred, edge_label)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.numel()
        total_examples += pred.numel()
    print(f"Epoch: {epoch:03d}, Loss: {total_loss / total_examples:.4f}")

Device: 'cuda'


  0%|                                                                                           | 0/43 [00:00<?, ?it/s]


ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'